In [1]:
%%time
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import wandb
import pandas as pd
from collections import defaultdict

# Table processing
def process_line(means, highlight, highlight_index, highlight_max, ignore_std):
    if highlight:
        if highlight_max:
            tops = set(means.groupby(highlight_index).idxmax())
        else:
            tops = set(means.groupby(highlight_index).idxmin())
    else:
        tops = set()

    def process_line(x):
        if ignore_std:
            if x.name in tops:
                return rf"\textbf{{{x['mean']:0.3f}}}"
            return rf"{x['mean']:0.3f}"
        if x.name in tops:
            return rf"\textbf{{{x['mean']:0.3f} $\pm$ {x['std']:0.3f}}}"
        return rf"{x['mean']:0.3f} $\pm$ {x['std']:0.3f}"

    return process_line


def mean_pm_std(
    data,
    index,
    columns,
    value,
    highlight=True,
    highlight_cols=True,
    highlight_max=True,
    ignore_std=False,
):
    assert len(data) > 0
    groupby = data.groupby([*index, *columns])
    means = groupby.mean()[value].rename("mean")
    stds = groupby.std()[value].rename("std")
    ddf = pd.concat([means, stds], axis=1).T
    highlight_index = columns if highlight_cols else index
    ddf = ddf.apply(
        process_line(means, highlight, highlight_index, highlight_max, ignore_std)
    )
    ddf = ddf.reset_index().pivot(index=index, columns=columns)
    ddf.columns = ddf.columns.droplevel(level=0)
    return ddf

    
def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def prepare_data(data):
    flattened_data = [flatten_dict(item) for item in data]
    return pd.DataFrame(flattened_data)



CPU times: user 1.07 s, sys: 265 ms, total: 1.33 s
Wall time: 7.84 s


In [47]:
api = wandb.Api(timeout=30)

# Project is specified by <entity/project-name>
runs = api.runs(
    "openproblems-comp/fast-tbg",
    filters={
        "$and": [
            {
                "created_at": {"$gt": "2025-01-26T00:00:00"},
                "tags": {"$in": ["eval"]},
                #'group': {'$in': ['5_vars']},
                # "config.data.n_particles": {"$eq": 22},
                #'config.model': {'$eq': model},
                #'config.lr': {'$lt': 1.01 * lr, '$gt': 0.99 * lr},
            }
        ]
    },
)

summary_list, config_list, name_list, tag_list = [], [], [], []
for run in runs:
    tag_list.append(run.tags)
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
tag_list = [str(t) for t in tag_list]
df = pd.concat(
    [
        pd.DataFrame(name_list, columns=["name"]),
        pd.DataFrame(tag_list, columns=["Tags"]),
        df_summary,
        df_config,
    ],
    axis=1,
)

In [54]:
import math


def filterer(x):
    if isinstance(x, float) and not math.isfinite(x):
        return False
    return "table" in list(x)


filtered_df = df[
    df["model/_target_"].isin(["src.models.flow_matching_module.FlowMatchLitModule"])
    & df["Tags"].apply(lambda x: "eval" in x)
    & df["Tags"].apply(lambda x: "v9" in x)
    # & df["Tags"].apply(lambda x: "table" in x)
    # & ~df["val/cropped_energy_w1"].isna()
    # & df["tags"].apply(filterer)
][
    [
        # "tags",
        "model/_target_",
        "model/net/_target_",
        "test/cropped_energy_w1",
        "test/resampled/cropped_energy_w1",
        # "val/effective_sample_size",
        "test/effective_sample_size",
        "data/n_particles",
        # "val/rama/torus_wasserstein",
        "test/rama/torus_wasserstein",
        "test/resampled/rama/torus_wasserstein",
        "test/samples_walltime",
        # "model/sampling_config/num_proposal_samples",
        # "model/sampling_config/num_test_proposal_samples",
    ]
].drop(
    columns=["model/_target_"]
)

# filtered_df.sort_values("data/n_particles")

In [55]:
renamed_df = filtered_df.replace(
    {
        "src.models.components.tbg.egnn_dynamics_ad2_cat.EGNN_dynamics_AD2_cat": "EQ-CFM",
        "src.models.components.dit.DIT3D": "DiT-CFM",
    }
).rename(columns={"model/net/_target_": "Model", "data/n_particles": "n_particles"})

In [56]:
renamed_df.groupby(["Model", "n_particles"]).count()

test/cropped_energy_w1  test/resampled/cropped_energy_w1  \
Model   n_particles                                                             
DiT-CFM 22.0                              2                                 2   
        33.0                              3                                 3   
        42.0                              4                                 4   
EQ-CFM  22.0                              3                                 3   
        33.0                              3                                 3   
        42.0                              4                                 4   

                     test/effective_sample_size  test/rama/torus_wasserstein  \
Model   n_particles                                                            
DiT-CFM 22.0                                  2                            2   
        33.0                                  3                            3   
        42.0                                  4                            4   
EQ-CFM  22.0                                  3                            3   
        33.0                                  3                            3   
        42.0                                  4                            4   

                     test/resampled/rama/torus_wasserstein  \
Model   n_particles                                          
DiT-CFM 22.0                                             2   
        33.0                                             3   
        42.0                                             4   
EQ-CFM  22.0                                             3   
        33.0                                             3   
        42.0                                             4   

                     test/samples_walltime  
Model   n_particles                         
DiT-CFM 22.0                             2  
        33.0                             3  
        42.0                             4  
EQ-CFM  22.0                             3  
        33.0                             3  
        42.0                             4

In [60]:
metrics = [
    # "test/resampled/cropped_energy_w1",
    # "test/effective_sample_size",
    # "test/cropped_energy_w1",
    # "test/rama/torus_wasserstein",
    "test/samples_walltime",
    # "test/resampled/rama/torus_wasserstein",
]
df_melt = renamed_df.dropna().melt(
    value_vars=metrics, id_vars=["Model", "n_particles"], var_name="Metric"
)
df_melt

,Model,n_particles,Metric,value
0,EQ-CFM,22.0,test/samples_walltime,7329.493164
1,EQ-CFM,22.0,test/samples_walltime,7996.049316
2,EQ-CFM,22.0,test/samples_walltime,7659.323730
3,EQ-CFM,33.0,test/samples_walltime,22269.408203
4,EQ-CFM,33.0,test/samples_walltime,22356.810547
5,EQ-CFM,42.0,test/samples_walltime,14507.354492
6,EQ-CFM,33.0,test/samples_walltime,22297.132812
7,DiT-CFM,22.0,test/samples_walltime,9749.890625
8,EQ-CFM,42.0,test/samples_walltime,14857.669922
9,DiT-CFM,33.0,test/samples_walltime,159.620850


In [59]:
results = mean_pm_std(
    df_melt, index=["n_particles", "Model"], columns=["Metric"], value="value", highlight=False
)
results = results[metrics]
results

Metric              test/effective_sample_size test/cropped_energy_w1  \
n_particles Model                                                       
22.0        DiT-CFM          0.013 $\pm$ 0.018  514.144 $\pm$ 719.296   
            EQ-CFM           0.233 $\pm$ 0.042      4.943 $\pm$ 1.147   
33.0        DiT-CFM          0.027 $\pm$ 0.007      0.760 $\pm$ 0.160   
            EQ-CFM           0.036 $\pm$ 0.027    856.417 $\pm$ 5.898   
42.0        DiT-CFM          0.025 $\pm$ 0.004      0.496 $\pm$ 0.134   
            EQ-CFM           0.091 $\pm$ 0.038      4.254 $\pm$ 1.102   

Metric              test/rama/torus_wasserstein      test/samples_walltime  
n_particles Model                                                           
22.0        DiT-CFM           1.763 $\pm$ 0.946  22821.139 $\pm$ 18485.536  
            EQ-CFM            1.155 $\pm$ 0.024     7661.622 $\pm$ 333.284  
33.0        DiT-CFM           0.887 $\pm$ 0.011        150.860 $\pm$ 7.879  
            EQ-CFM            1.802 $\pm$ 0.018     22307.784 $\pm$ 44.664  
42.0        DiT-CFM           1.763 $\pm$ 0.074        191.740 $\pm$ 9.798  
            EQ-CFM            2.453 $\pm$ 0.045    14620.535 $\pm$ 228.039

In [53]:
print(
    results.to_latex(
        float_format="{:.3f}".format,
    )
)

\begin{tabular}{lllll}
\toprule
 & Metric & test/effective_sample_size & test/cropped_energy_w1 & test/rama/torus_wasserstein \\
n_particles & Model &  &  &  \\
\midrule
42.000000 & EQ-CFM & 0.123 $\pm$ 0.006 & 4.229 $\pm$ 1.284 & 2.414 $\pm$ 0.000 \\
\cline{1-5}
\bottomrule
\end{tabular}

